This notebook estimates domestic production domestically consumed for food items and for forestry goods, based on the production data of FAOSTAT.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('C://Users/11max/PycharmProjects/Regioinvent/src/')
import regioinvent
import sqlite3

Connect to both SQL databases (BACI with all the data and the filtered version)

In [2]:
# load formatted BACI trade data
conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/trade_data.db')

full_baci_conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/baci_trade_data_full_version.db')

# Agriculture

In [41]:
# read FAOSTAT data
faodata = pd.read_csv('SUA_Crops_Livestock_E_All_Data_(Normalized).csv', low_memory=False)
# only keep production data for relevant years
faodata = faodata.loc[faodata.Element == 'Production'].loc[faodata.Year.isin([2020, 2021, 2022, 2023, 2024])]
# only keep relevant columns
faodata = faodata.loc[:, ['Area', 'Item Code', 'Item', 'Year', 'Unit', 'Value']]

# change to country codes ISO
countries = pd.read_excel('C://Users/11max/PycharmProjects/Regioinvent/Data/Production/FAOSTAT/FOASTAT_country_mapping.xlsx').set_index('Area')
faodata.loc[:, 'Country'] = [countries.loc[i, 'ISO code'] if i in countries.index else 'N/A' for i in faodata.Area]
faodata = faodata[faodata.loc[:, 'Country'] != 'N/A']
faodata = faodata.drop('Area', axis=1)

In [42]:
# load mapping between FAOSTAT and the products within regioinvent
mapping_faostat = pd.read_excel('FAOSTAT_products_to_map_updated.xlsx', dtype='str')
# item codes as integers to merge with faodata dataframe
mapping_faostat.loc[:, 'Item Code'] = mapping_faostat.loc[:, 'Item Code'].astype('int')
# add mapping to faodata dataframe
faodata = faodata.merge(mapping_faostat, left_on=['Item Code', 'Item'], right_on=['Item Code', 'Item'], how='left')
# stack() to concatenate all possible HS codes together
faodata = faodata.set_index(['Item Code', 'Item', 'Year', 'Unit', 'Value', 'Country']).stack().droplevel(-1).reset_index()
faodata = faodata.rename(columns={0: 'HS codes'})

FAOSTAT production data within the SUA accounts corresponds to total production. Because we already cover exports through BACI, we are not interested in the total production values, but rather into the domestic consumption of domestically-produced commodities only, i.e., total production - whatever was exported. We thus require exports to subtract from total production, hence why we load BACI data here. Furhtermore, we load the full version as some commodities of FAOSTAT include more than just corresponding ecoinvent commodities. For instance, even if in ecoinvent there is only data in "wheat", the "Wheat" category of FAOSTAT also include "durum wheat". Therefore, we need to load the export data for wheat and for durum wheat, to be able to correctly estimate the residual domestic production of the "Wheat" item of FAOSTAT.

In addition, to separate between, e.g., wheat and durum wheat, we rely on this export data. If Canada exports 80% of wheat and 20% of durum wheat, we assume that the domestic production of Canada also follows this 80/20 ratio.

In [21]:
# extract relevant export values from BACI
required_hs_export_values = set(faodata.loc[:,'HS codes'])

placeholders = ', '.join([f"'{str(val)}'" for val in required_hs_export_values])

query = f"SELECT * FROM [International trade data] WHERE cmdCode IN ({placeholders})"
data = pd.read_sql(query, full_baci_conn)

# determine net exports
import_data = data.set_index(['cmdCode', 'refYear', 'importer', 'exporter']).drop(
    ['value (1,000 USD)'], axis=1).groupby(['cmdCode', 'refYear', 'importer']).sum().reset_index()
export_data = data.set_index(['cmdCode', 'refYear', 'exporter']).drop(
    ['value (1,000 USD)', 'importer'], axis=1).groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()
import_data = import_data.rename(columns={'importer': 'country', 'quantity (t)': 'import_qty'})
export_data = export_data.rename(columns={'exporter': 'country', 'quantity (t)': 'export_qty'})

# outer merge to ensure we don't lose data 
net_exports_data = pd.merge(
    export_data, 
    import_data, 
    on=['cmdCode', 'refYear', 'country'], 
    how='outer'
)

# fill NaNs with 0 (so subtraction works if one side is missing)
net_exports_data[['export_qty', 'import_qty']] = net_exports_data[['export_qty', 'import_qty']].fillna(0)

# calculate Net Exports
net_exports_data['net_export'] = net_exports_data['export_qty'] - net_exports_data['import_qty']

net_exports_data = net_exports_data[['cmdCode', 'refYear', 'country', 'net_export']]

# remove negative net export balance
net_exports_data = net_exports_data[net_exports_data.loc[:,'net_export'] > 0].dropna()

Note here that there are two specific cases. 1) When a commodity does not have export data reliably on all selected years (i.e., there are data points for 2020 and 2023, but nothing for 2021 and 2022) then the average of all available years is used for missing years. 2) WHen there is simply no reported export data for a given commodity (maybe because the commodity is only consumed domestically and not exported at all) then we calculate a global average of exports between the commodities to separate and apply this split to estimate the split of the domestic production. So if Brazil does not export any durum wheat or wheat at all, but that overall globally wheat represent 60% of exports vs 40% for durum wheat, then we assume that Brazil's domestic production is composed of 60% of wheat and 40% of durum wheat.

In [43]:
# Calculate total global export volume for every HS code
global_hs_totals = net_exports_data.groupby('cmdCode')['net_export'].sum().reset_index()
global_hs_totals.columns = ['HS codes', 'global_export_volume']

for commodity in set(faodata.Item):
    df = faodata.loc[faodata.Item == commodity]
    codes = set(df.loc[:, 'HS codes'])
    if len(codes) > 1:
        # calculate total exports for the commodities
        total_export_cmd = net_exports_data.loc[net_exports_data.cmdCode.isin(codes)].drop('cmdCode', axis=1).groupby(['refYear', 'country']).sum().reset_index()
        total_export_cmd = total_export_cmd.rename(columns={'net_export': 'total_net_export'})
        # filter the net exports data for only the relevant commodities
        net_exports_cmd = net_exports_data.loc[net_exports_data.cmdCode.isin(codes)]
        # calculate the distribution shares of each commodity in each country, based on export shares
        net_exports_cmd = net_exports_cmd.merge(total_export_cmd, on=['refYear', 'country'], how='outer')
        net_exports_cmd.loc[:, 'net_export_share'] = net_exports_cmd.net_export / net_exports_cmd.total_net_export
        # merge this export share data with production data
        dff = df.merge(net_exports_cmd.drop(['net_export', 'total_net_export'], axis=1), 
                       left_on=['HS codes', 'Year', 'Country'], 
                       right_on=['cmdCode', 'refYear', 'country'], how='left').drop(['cmdCode', 'refYear', 'country'], axis=1)
        # fill with 0 if the sum of shares between the different commodities equals to 1
        current_group_sums = dff.groupby(['Item Code', 'Item', 'Year', 'Country', 'Unit', 'Value'])['net_export_share'].transform('sum')
        is_group_complete = np.isclose(current_group_sums, 1.0, atol=1e-3)
        dff['net_export_share'] = np.where(is_group_complete & dff['net_export_share'].isna(), 0.0, dff['net_export_share'])
        # for countries where for a few years no export data was available, but for some other years yes, we apply the average over the studied years
        average_shares = dff.groupby(['Country', 'Unit', 'Value', 'Year', 'HS codes'])['net_export_share'].transform('mean')
        dff['net_export_share'] = dff['net_export_share'].fillna(average_shares)
        # for countries where there are no export data available at all, we apply global average export shares between the commodities
        commodity_hs_mapping = dff[['Item', 'Item Code', 'HS codes']].drop_duplicates()
        global_shares = pd.merge(commodity_hs_mapping, global_hs_totals, on='HS codes', how='left').fillna(0)
        global_commodity_total = global_shares.groupby(['Item', 'Item Code'])['global_export_volume'].transform('sum')
        global_shares['global_share'] = np.where(
            global_commodity_total > 0,
            global_shares['global_export_volume'] / global_commodity_total,
            1 / global_shares.groupby(['Item', 'Item Code'])['HS codes'].transform('count') # Hard fallback if code never appears globally
        )
        # Create a clean dictionary/mapping for quick lookup
        global_share_map = global_shares.set_index(['Item', 'Item Code', 'HS codes'])['global_share'].to_dict()
        remaining_nans = dff['net_export_share'].isna()
        if remaining_nans.any():
            nan_pairs = dff.loc[remaining_nans, ['Item', 'Item Code', 'HS codes']].to_records(index=False)
            dff.loc[remaining_nans, 'net_export_share'] = [global_share_map.get(tuple(pair), 0) for pair in nan_pairs]
        # apply the net export ratios to production values to split between the commodities
        dff.loc[:, 'Value'] *= dff.loc[:, 'net_export_share']
        dff = dff.drop('net_export_share', axis=1)
        # replace old values with the new ones
        faodata = faodata.drop(faodata.loc[faodata.Item == commodity].index)
        faodata = pd.concat([faodata, dff])
        
faodata = faodata.drop(['Item Code', 'Item', 'Unit'], axis=1)

In [44]:
# for HS codes linked to multiple FAOSTAT commodities, we simply sum their total production value
faodata = faodata.groupby(['Year', 'Country', 'HS codes']).sum().reset_index()

In [45]:
# merge to then subtract net exports from total production value
faodata = faodata.merge(net_exports_data, left_on=['HS codes', 'Year', 'Country'], right_on=['cmdCode', 'refYear', 'country'], how='left').loc[:, ['Year', 'HS codes', 'Value', 'net_export', 'Country']]
faodata.loc[:, 'Value'] = faodata.loc[:, 'Value'] - faodata.loc[:, 'net_export'].fillna(0)
# only keep positive residual production values
faodata = faodata.drop('net_export', axis=1)
faodata = faodata[faodata.Value > 0].fillna(0)

Format the data

In [47]:
faodata = faodata.rename(columns={'Year': 'refYear', 'Value':'quantity (t)', 'Country':'producer', 'HS codes':'cmdCode'})
faodata = faodata.T.reindex(['cmdCode', 'refYear', 'producer', 'quantity (t)']).T
faodata = faodata.reset_index().drop('index',axis=1)
faodata.loc[:, 'source'] = 'FAOSTAT - SUA_Crops_Livestock_E_All_Data_(Normalized).csv - taken from FAOSTAT website in May 2026'
faodata.loc[:,'country_coverage'] = 'complete'

Add the data to the SQL database inside the "Real production" data table

# Forestry

For Forestry data, there is one significant change. The production data this time does not equal to total production data, but rather to the part of the domestic production that is consumed domestically, i.e., the number we want directly. There is thus no need to subtract exports from the production value.

However, we still rely on export data to split HS commodities whenever the FOASTAT category is more aggregated.

In [50]:
# format FAOSTAT data
faodata = pd.read_csv('Forestry_E_All_Data.csv', low_memory=False)
faodata = faodata.loc[faodata.Element == 'Production'].loc[:, ['Area', 'Item Code', 'Item', 'Unit', 'Element', 'Y2020', 'Y2021', 'Y2022', 'Y2023', 'Y2024']]

# change to country codes ISO
countries = pd.read_excel('C://Users/11max/PycharmProjects/Regioinvent/Data/Production/FAOSTAT/FOASTAT_country_mapping.xlsx')
countries = countries.set_index('Area')
faodata.loc[:, 'country code'] = [countries.loc[i, 'ISO code'] if i in countries.index else 'N/A' for i in faodata.Area]
faodata = faodata[faodata.loc[:,'country code'] != 'N/A']

In [51]:
# load mapping between FAOSTAT and the products within regioinvent
mapping_faostat = pd.read_excel('FAOSTAT_forestry_products_to_map.xlsx', dtype=str).set_index('Item Code')
mapping_faostat = mapping_faostat.drop('Item', axis=1).stack().droplevel(1)

In [19]:
# extract relevant export values from BACI
required_hs_export_values = set(mapping_faostat.tolist())

placeholders = ', '.join([f"'{str(val)}'" for val in required_hs_export_values])

query = f"SELECT * FROM [International trade data] WHERE cmdCode IN ({placeholders})"
data = pd.read_sql(query, full_baci_conn)

# determine net exports
import_data = data.set_index(['cmdCode', 'refYear', 'importer', 'exporter']).drop(
    ['value (1,000 USD)'], axis=1).groupby(['cmdCode', 'refYear', 'importer']).sum().reset_index()
export_data = data.set_index(['cmdCode', 'refYear', 'exporter']).drop(
    ['value (1,000 USD)', 'importer'], axis=1).groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()
import_data = import_data.rename(columns={'importer': 'country', 'quantity (t)': 'import_qty'})
export_data = export_data.rename(columns={'exporter': 'country', 'quantity (t)': 'export_qty'})

# outer merge to ensure we don't lose data 
net_exports_data = pd.merge(
    export_data, 
    import_data, 
    on=['cmdCode', 'refYear', 'country'], 
    how='outer'
)

# fill NaNs with 0 (so subtraction works if one side is missing)
net_exports_data[['export_qty', 'import_qty']] = net_exports_data[['export_qty', 'import_qty']].fillna(0)

# calculate Net Exports
net_exports_data['net_export'] = net_exports_data['export_qty'] - net_exports_data['import_qty']

net_exports_data = net_exports_data[['cmdCode', 'refYear', 'country', 'net_export']]

# remove negative net export balance
net_exports_data = net_exports_data[net_exports_data.loc[:,'net_export'] > 0].dropna()

In [52]:
master_df = pd.DataFrame()

# itreate through FAO item, e.g., 1861 or 1864
for item in set(faodata.loc[:, 'Item Code']):
    # check if FAO item has been mapped or not
    if str(item) in mapping_faostat.index:
        # extract HS code(s) linked to FAO item code
        conc_HS = mapping_faostat.loc[str(item)]
        # if it's a 1 for 1 match, e.g., 1861 -> 4403
        if type(conc_HS) == str:
            # then loop through countries
            for cty in set(faodata.loc[:, 'country code']):
                df = faodata.loc[(faodata.loc[:, 'Item Code'] == item) & (faodata.loc[:, 'country code'] == cty)].copy()
                # loop through years
                for year in [2020, 2021, 2022, 2023, 2024]:
                    if not df.loc[:, 'Y'+str(year)].empty:
                        # and simply report the value in the master_df
                        master_df = pd.concat([master_df, 
                                               pd.DataFrame([df.loc[:, 'Y'+str(year)].iloc[0], cty, conc_HS, year], 
                                                            index=['quantity (t)', 'producer', 'cmdCode', 'refYear']).T])
                    
        # if it's not a 1 for 1 match, e.g., 1864 -> 440139 & 440111
        else:
            # calculate the shares of each HS commodity based on net exports
            net_exports_item = net_exports_data.loc[net_exports_data.cmdCode.isin(conc_HS)].copy()
            group_totals = net_exports_item.groupby(['country', 'refYear'])['net_export'].transform('sum')
            net_exports_item.loc[:, 'share'] = net_exports_item.loc[:, 'net_export'] / group_totals
            # loop through countries
            for cty in set(faodata.loc[:, 'country code']):
                df = faodata.loc[(faodata.loc[:, 'Item Code'] == item) & (faodata.loc[:, 'country code'] == cty)].copy()
                # loop through years
                for year in [2020, 2021, 2022, 2023, 2024]:
                    if not df.loc[:, 'Y'+str(year)].empty:
                        for HS_item in conc_HS:
                            # if there is available data for that commodity/country/year combination in the export data
                            try:
                                share_HS_item = net_exports_item.loc[
                                (net_exports_item.country == cty) & (net_exports_item.refYear == year) & (net_exports_item.cmdCode == HS_item), 'share'].iloc[0]
                            except IndexError:
                                continue
                            # then copy the df
                            dff = pd.DataFrame([df.loc[:, 'Y'+str(year)].iloc[0], cty, HS_item, year], 
                                                            index=['quantity (t)', 'producer', 'cmdCode', 'refYear']).T
                            # and apply the export ratio to the production value
                            dff.loc[:, 'quantity (t)'] *= share_HS_item
                            # dump into master_df
                            master_df = pd.concat([master_df, dff])

100%|██████████████████████████████████████████████████████████████████████████████████| 78/78 [00:22<00:00,  3.48it/s]


In [53]:
# groupby those who appear in multiple mappings (like 4403)
master_df = master_df.groupby(['producer','cmdCode','refYear']).sum().reset_index()

Some of data from FAOSTAT are provided as cubic meters while all the physical data in BACI is in tonnes. We thus estimated densities for the different FAOSTAT items (identified by their numerical codes) relying on the LLM Gemini3. Note that wood chpis and residues have a significantly lower density due to the air included in the chips volume.

In [54]:
densities = { # in t/m3
    '1861': 0.7,
    '1864': 0.7,
    '1865': 0.7,
    '1866': 0.5,
    '1867': 0.75,
    '1868': 0.7,
    '1601': 0.5,
    '1604': 0.75,
    '1871': 0.7,
    '1623': 0.5,
    '1626': 0.75,
    '1872': 0.63,
    '1632': 0.48,
    '1633': 0.68,
    '1640': 0.6,
    '1697': 0.65,
    '1619': 0.25,
    '1620': 0.2,
    '1606': 0.65,
    '1874': 0.8,
    '1647': 0.95,
    '1648': 0.8,
    '1636': 0.5,
    '1688': 0.5
}

In [55]:
# use densities to convert quantity in m3 (from FAOSTAT) to quantity in tonnes (to be able to match with BACI export/import data)
for code in set(master_df.cmdCode):
    # check if FAO data is in m3 for this code
    FAO_item = mapping_faostat.loc[mapping_faostat == code].index[0]
    if FAO_item in densities:
        master_df.loc[master_df.cmdCode == code, 'quantity (t)'] *= densities[FAO_item]

In [56]:
# Guadeloupe, French Guyana and Martinique should be tagged as France to match to BACI correctly
master_df.loc[master_df.producer == 'GLP', 'producer'] = 'FRA'
master_df.loc[master_df.producer == 'GUF', 'producer'] = 'FRA'
master_df.loc[master_df.producer == 'MTQ', 'producer'] = 'FRA'
master_df = master_df.groupby(['producer', 'cmdCode', 'refYear']).sum().reset_index()

In [61]:
master_df.loc[:, 'source'] = 'FAOSTAT - Forestry_E_All_Data.csv - taken from FAOSTAT website in May 2026. m3 converted to tonnes through densities from Gemini3.'
master_df.loc[:, 'country_coverage'] = 'complete'

Add the data to the SQL database inside the "Real production" data table